# 대화 상태 (Conversation State)
모델 상호작용 중 대화 상태를 관리하는 방법을 알아보세요.
OpenAI는 대화 상태를 관리하는 몇 가지 방법을 제공하며, 이는 대화에서 여러 메시지 또는 차례에 걸쳐 정보를 보존하는 데 중요합니다.

각 텍스트 생성 요청은 독립적이고 상태가 지정되지 않지만(Assistants API를 사용하는 경우 제외), 텍스트 생성 요청에 추가 메시지를 매개변수로 제공하여 여러 차례에 걸친 대화를 구현할 수 있습니다. 

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-4.1-nano"

### 대화 상태 수동 관리

In [3]:
response = client.chat.completions.create(
    model=Model,
    messages=[
        {"role": "user", "content": "똑! 똑!"},
        {"role": "assistant", "content": "누구십니까?"},
        {"role": "user", "content": "배달 왔습니다."},
    ],
)

print(response.choices[0].message.content)

네, 감사합니다. 바로 받아가겠습니다!


사용자 메시지와 어시스턴트 메시지를 번갈아 사용하면 모델에 대한 한 번의 요청으로 대화의 이전 상태를 캡처할 수 있습니다.  
생성된 응답 간에 컨텍스트를 수동으로 공유하려면 모델의 이전 응답 출력을 입력으로 포함하고 해당 입력을 다음 요청에 추가하세요.  
이렇게 이전 응답을 새 요청에 추가하면 대화가 자연스럽게 느껴지고 이전 상호작용의 컨텍스트가 유지됩니다.

In [4]:
history = [
    {
        "role": "user", "content": "재미있는 농담을 해줘."
    }
]

response = client.chat.completions.create(
    model=Model,
    messages=history,
)

print(response.choices[0].message.content)

history.append(response.choices[0].message)
history.append({ "role": "user", "content": "한가지 더 해줘." })

second_response = client.chat.completions.create(
    model=Model,
    messages=history,
)

print(second_response.choices[0].message.content)

물론이죠! 여기 재미있는 농담 하나 드릴게요:

왜 바다가 늘 슬픈지 아세요?  
왜냐하면 파도(파도) 때문에 마음이 흔들리거든요!
물론이죠! 또 하나 더 드릴게요:

왜 컴퓨터는 바다를 싫어할까요?  
바다가 너무 "네트워크"에 침범해서요!


### 대화 상태용 OpenAI API
response API를 사용하면 대화 상태를 자동으로 더 쉽게 관리할 수 있으므로 대화가 진행될 때마다 수동으로 입력을 전달할 필요가 없습니다.  
Responses API는 상태를 저장하므로 대화 전반의 컨텍스트 관리가 간단한 매개변수로 가능합니다.

In [5]:
response = client.responses.create(
    model=Model,
    input="재미있는 농담을 해줘.",
)
print(response.output_text)

second_response = client.responses.create(
    model=Model,
    previous_response_id=response.id,
    input=[{"role": "user", "content": "그게 왜 재미 있는지 설명해줘"}],
)
print(second_response.output_text)

좋아! 여기 농담 하나 해줄게:

왜 수학책이 슬퍼했을까요?  
답은... 너무 많은 문제들이 있어서요!
물론이죠! 이 농담이 재미있는 이유는 말장난에 기반하고 있기 때문이에요. "문제"라는 단어는 두 가지 의미를 가지고 있는데요:

1. 수학에서 "문제"는 해결해야 하는 질문이나 과제.
2. 일상생활에서 "문제"는 고민거리나 어려운 상황.

이 농담에서는 수학책이 "너무 많은 문제들" 때문에 슬퍼한다고 했는데, 여기서 "문제"는 수학적 문제뿐 아니라, 어려움이나 고민거리도 의미하니까, 웃음을 유발하는 의도적인 이중 의미가 있어요! 그래서 웃기게 느껴지는 것이죠.


### 컨텍스트 창 관리
컨텍스트 창을 이해하면 스레드된 대화를 성공적으로 생성하고 모델 상호작용 전반의 상태를 관리하는 데 도움이 됩니다.   
컨텍스트 창은 단일 요청에서 사용할 수 있는 최대 토큰 수입니다. 이 최대 토큰 수에는 입력, 출력 및 추론 토큰이 포함됩니다. 모델의 컨텍스트 창에 대한 자세한 내용은 모델 세부 정보를 참조하세요. 

다음 토큰 수가 컨텍스트 창 총계에 적용됩니다.
- 입력 토큰( Responses APIinput 에 대한 배열 에 포함하는 입력)  
- 출력 토큰(프롬프트에 대한 응답으로 생성된 토큰)  
- 추론 토큰(모델이 응답을 계획하는 데 사용됨)
  
컨텍스트 창 제한을 초과하여 생성된 토큰은 API 응답에서 잘릴 수 있습니다.

<img src="https://i.imgur.com/WcWPlmZ.png" width=500 />

### Reasoning (추론)

o3 및 o4-mini와 같은 추론 모델은 강화 학습을 통해 추론을 수행하도록 훈련된 LLM입니다. 추론 모델은 사용자에게 응답하기 전에 먼저 생각하여 , 사용자에게 응답하기 전에 긴 내부 사고 사슬을 생성합니다. 추론 모델은 복잡한 문제 해결, 코딩, 과학적 추론, 그리고 에이전트 워크플로우를 위한 다단계 계획 수립에 탁월합니다. 또한 , 경량 코딩 에이전트인 Codex CLI 에 가장 적합한 모델입니다.

In [6]:
prompt = """
문자열로 표현된 행렬을 '[1,2],[3,4],[5,6]' 형식으로 받아서 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트를 작성하세요.
"""

response = client.responses.create(
    model="o4-mini",
    reasoning={"effort": "medium"},
    input=[
        {
            "role": "user", 
            "content": prompt
        }
    ]
)

print(response.output_text)

아래와 같이 작성하시면 됩니다. `ast.literal_eval` 을 이용해서 문자열을 파이썬 리스트로 안전하게 파싱하고, `zip` 으로 전치한 뒤 다시 원래 포맷의 문자열로 조합합니다.

```python
#!/usr/bin/env python3
import ast

def transpose_matrix_str(matrix_str: str) -> str:
    # 입력이 '[1,2],[3,4],[5,6]' 형태이므로 양쪽에 대괄호를 하나씩 더 붙여서 literal_eval 로 파싱
    matrix = ast.literal_eval(f'[{matrix_str}]')  # => [[1,2],[3,4],[5,6]]
    # 전치
    transposed = list(zip(*matrix))              # => [(1,3,5),(2,4,6)]
    # 다시 '[1,3,5],[2,4,6]' 형태의 문자열로 조합
    return ','.join(
        '[' + ','.join(str(x) for x in row) + ']'
        for row in transposed
    )

if __name__ == '__main__':
    s = input().strip()
    print(transpose_matrix_str(s))
```

사용 예시:

```
$ python3 transpose.py
[1,2],[3,4],[5,6]
[1,3,5],[2,4,6]
```

— 입력 문자열이 `[a,b],[c,d],…` 형태라면 위 스크립트가 전치행렬을 동일한 포맷으로 출력해 줍니다.


In [7]:
import ast

def transpose_str_matrix(s: str) -> str:
    # 문자열 "[1,2],[3,4],[5,6]" → "[[1,2],[3,4],[5,6]]" 로 감싸서 literal_eval
    matrix = ast.literal_eval(f'[{s}]')
    # 전치
    transposed = list(map(list, zip(*matrix)))
    # "[[1,3,5],[2,4,6]]" → "[1,3,5],[2,4,6]"
    return ','.join('[' + ','.join(map(str, row)) + ']' for row in transposed)

s = "[1,2],[3,4],[5,6]"
print(transpose_str_matrix(s))

[1,3,5],[2,4,6]


### Python 프로젝트 계획 및 생성

In [8]:
prompt = """
사용자 질문을 받아 답변과 매핑된 데이터베이스에서 검색하는 Python 앱을 만들고 싶습니다.
유사한 답변이 있으면 일치하는 답변을 검색합니다.
만약 일치하지 않으면 사용자에게 답변을 입력하도록 요청하고
질문/답변 쌍을 데이터베이스에 저장합니다. 필요한 디렉터리 구조를 계획한 후
각 파일을 전체적으로 반환합니다.
코드 전체가 아닌 시작과 끝에만 해설을 제공하세요.
"""

response = client.responses.create(
    model="o4-mini",
    input=[
        {
            "role": "user",
            "content": prompt,
        }
    ]
)

print(response.output_text)

아래와 같이 프로젝트를 구성해 보세요.

시작 전 설명
- SQLite를 이용해 질문/답변을 저장합니다.
- difflib의 SequenceMatcher로 간단한 유사도 매칭을 합니다.
- 유사 질문이 없으면 사용자에게 답변 입력을 요청하고 DB에 저장합니다.

프로젝트 디렉터리 구조
qa_app/
  src/
    db.py
    similarity.py
    main.py
  data/           ← 빈 디렉터리(qa.db는 실행 시 생성)
  requirements.txt

──────────────────────────────────────────
파일: src/db.py
──────────────────────────────────────────
import sqlite3
import os

DB_PATH = os.path.join(os.path.dirname(__file__), '..', 'data', 'qa.db')

def get_connection():
    db_dir = os.path.dirname(DB_PATH)
    if not os.path.exists(db_dir):
        os.makedirs(db_dir)
    return sqlite3.connect(DB_PATH)

def init_db():
    conn = get_connection()
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS qa_pairs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            question TEXT,
            answer TEXT
        )
    ''')
    conn.commit()
    conn.close()

def get_all_pairs():
    conn = get_connection()
    c = conn.cursor()
    c.